## Current endpoint IK: SE(3) with bounded joints
The endpoint formulation and export are defined in the final section of this notebook. That section can be run independently of the differential-QP comparison below. It uses world position, a unit quaternion and four bounded joints, exact tool-pose constraints, and secondary posture/movement and arm-base alignment preferences. Runtime supplies joint bounds and the alignment weight, uses deterministic multistart, and validates the endpoint with FK. An IK endpoint does not certify a collision-free path.


In [1]:
import sys
import os
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir)
import system.robot as robotSystem
import casadi as ca
import system.utils.transformation_matrix as Transformation
import numpy as np

In [2]:
alpha = robotSystem.RobotDynamics()
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
path_to_urdf = os.path.join(
    project_root,
    'usage',
    'urdf','reach_alpha_5',
    'alpha_5_robot.urdf'
)
alpha.from_file(path_to_urdf)
root = "base_link"
tip = "alpha_standard_jaws_base_link"
kinematic_dict, K, P, L, D, C, g, B, qdd, joint_torque, sys_id_coeff, F_payload = alpha.build_model(root, tip, floating_base=True)
kinematic_dict = alpha.get_kinematic_dict()
c_parms, m_params, I_params, fv_coeff, fc_coeff, fs_coeff, v_s_coeff, vec_g, r_com_body, m_p, q, q_dot, q_dotdot, tau, base_pose, world_pose, tip_offset_pose = kinematic_dict['parameters']
n_joints = alpha.get_n_joints(root, tip)

In [3]:
# decision variables for IK, these are spatial velocities, not Euler rates
twist_base_W = ca.SX.sym('twist_base_W', 6, 1)  
v_base_W     = twist_base_W[0:3] # base linear velocity in world
omega_base_W = twist_base_W[3:6] # base angular velocity in world

q_dot_cmd = ca.SX.sym('q_dot_cmd', n_joints, 1)
nu = ca.vertcat(twist_base_W, q_dot_cmd)

# position task
Jp_ee_W = kinematic_dict['geo_J_full'][-1][0:3, :]    # linear Jacobian (world)
p_ee_W  = kinematic_dict['Fks'][-1][0:3]              # EE position in world

kp = ca.SX.sym('kp', 3, 1)
p_des = ca.SX.sym('p_des', 3, 1)

# desired end-effector linear velocity (per-axis gains)
v_des = ca.diag(kp) @ (p_des - p_ee_W)                # 3 x 1

# task residual: match task-space linear velocity
e_p_task = Jp_ee_W @ nu - v_des                       # 3 x 1

In [4]:
# build a full desired rotation and use a rotation error
# Desired tool axes expressed in world frame
a_des_x = ca.SX.sym('a_des_x', 3, 1)
a_des_y = ca.SX.sym('a_des_y', 3, 1)
a_des_z = ca.SX.sym('a_des_z', 3, 1)

# Normalize desired axes
x_d = a_des_x / (ca.norm_2(a_des_x) + 1e-12)
y_d = a_des_y / (ca.norm_2(a_des_y) + 1e-12)
z_d = a_des_z / (ca.norm_2(a_des_z) + 1e-12)

# Orthonormalize, prioritize z, then x, then y (Gram Schmidt)
x_d = x_d - z_d * (z_d.T @ x_d)
x_d = x_d / (ca.norm_2(x_d) + 1e-12)

# Build a right handed orthonormal frame
y_d = ca.cross(z_d, x_d)
y_d = y_d / (ca.norm_2(y_d) + 1e-12)
x_d = ca.cross(y_d, z_d)

# Desired tool orientation in world, columns are tool axes
R_des = ca.horzcat(x_d, y_d, z_d)

# Current tool orientation in world
R_ee_W = kinematic_dict['R_symx'][-1]

# Spatial orientation error in world frame
E = R_ee_W @ R_des.T - R_des @ R_ee_W.T
e_R_W = 0.5 * ca.vertcat(E[2,1], E[0,2], E[1,0])

# Convert orientation error into desired angular velocity
k_axis = ca.SX.sym('k_axis')
w_des_axis = -k_axis * e_R_W

# Angular Jacobian mapping nu to tool angular velocity in world
Jw_ee_W = kinematic_dict['geo_J_full'][-1][3:6, :]

# Residual enforcing the orientation task
e_axis_task = Jw_ee_W @ nu - w_des_axis

In [ ]:
# roll,pitch soft stabilization (servo Euler angles back toward 0)
phi   = world_pose[3]
theta = world_pose[4]
psi = world_pose[5]
k_rp = ca.SX.sym('k_rp')                         # scalar

# desired Euler rates (stabilize roll, pitch, leave yaw free)
phi_dot_des   = k_rp * (0 - phi)
theta_dot_des = k_rp * (0 - theta)
psi_dot_des   = 0

# World angular velocity for URDF R = Rz(yaw) Ry(pitch) Rx(roll).
# Columns are the world axes of instantaneous roll, pitch and yaw rotation.
Ts = ca.horzcat(
    ca.vertcat(ca.cos(psi)*ca.cos(theta), ca.sin(psi)*ca.cos(theta), -ca.sin(theta)),
    ca.vertcat(-ca.sin(psi), ca.cos(psi), 0),
    ca.vertcat(0, 0, 1))
euler_dot_des = ca.vertcat(phi_dot_des, theta_dot_des, psi_dot_des)
omega_des_rp  = Ts @ euler_dot_des                        # 3x1, desired world angular velocity

# residual in angular velocity space (consistent with geo Jw)
e_omega = omega_base_W - omega_des_rp   # 3x1

# only penalize roll and pitch behavior, do not fight yaw
e_rp    = e_omega[0:2]                  # 2x1

In [6]:
# relative orientation task between the floating base and the arm’s first link to keep it centered
a_x = ca.DM([1, 0, 0])                      # local x axis

R_B_W  = Transformation.rot_from_rpy(phi, theta, psi)     # base rotation in world
R_J0_W = kinematic_dict['R_symx'][0]                      # joint0 rotation in world

x_B_W  = R_B_W  @ a_x                      # base x axis in world
x_J0_W = R_J0_W @ a_x                     # joint0 x axis in world

Jw_J0_W    = kinematic_dict['geo_J_full'][0][3:6, :]  # joint0 angular Jacobian
omega_J0_W = Jw_J0_W @ nu                 # joint0 angular velocity in world

omega_rel = omega_J0_W - omega_base_W    # joint0 relative to base

e_align = ca.cross(x_J0_W, -x_B_W)        # axis misalignment error
k_align = ca.SX.sym('k_align')
w_des_align = k_align * e_align           # desired relative angular velocity

e_align_task = omega_rel - w_des_align    # alignment residual

In [7]:
# regularization (keeps solution small, avoids base thrashing)
e_reg = nu                                       # (6+n) x 1

In [8]:
# weighted least squares objective
w_rp  = ca.SX.sym('w_rp')                        # scalar
w_reg = ca.SX.sym('w_reg')                       # scalar
w_axis = ca.SX.sym('w_axis')  # scalar
w_align = ca.SX.sym('w_align')
cost = (
    ca.dot(e_p_task, e_p_task)
    + w_axis * ca.dot(e_axis_task, e_axis_task)
    + w_rp  * ca.dot(e_rp, e_rp)
    + w_reg * ca.dot(e_reg, e_reg)
    + w_align * ca.dot(e_align_task, e_align_task)
    
)

In [ ]:
# 1. Compute Hessian and Full Gradient
H, g_full = ca.hessian(cost, nu)

# 2. Extract the linear term of the gradient
# The full gradient is (H * nu + g_linear). 
# By substituting nu=0, we isolate g_linear.
g_linear = ca.substitute(g_full, nu, ca.DM.zeros(nu.shape))

# Bound every integrated step, not merely the returned endpoint.
# Runtime limits come from the deployed robot_description, not this notebook.
dt = ca.SX.sym('dt')
q_lower = ca.SX.sym('q_lower', n_joints)
q_upper = ca.SX.sym('q_upper', n_joints)
dq_max = ca.SX.sym('dq_max', n_joints)
lam = 1e-8 + 1e-6 * w_reg
H_regularized = H + lam * ca.SX.eye(H.size1())
qp = ca.conic('bounded_ik_qp', 'qrqp',
    {'h': H_regularized.sparsity(), 'a': ca.Sparsity(0, nu.numel())},
    {'print_header': False, 'print_iter': False, 'error_on_fail': True})
lower_rate = ca.fmax(-dq_max, (q_lower - q) / dt)
upper_rate = ca.fmin(dq_max, (q_upper - q) / dt)
nu_star = qp(h=H_regularized, g=g_linear,
    lbx=ca.vertcat(ca.DM.ones(6)*-ca.inf, lower_rate),
    ubx=ca.vertcat(ca.DM.ones(6)*ca.inf, upper_rate))['x']


In [ ]:


# linear part integrates directly
p_world_next = world_pose[0:3] + nu_star[0:3] * dt

# convert omega -> euler_dot for integration
# Reuse the URDF world-angular-velocity map Ts defined above.

euler_dot_star = ca.solve(Ts, nu_star[3:6])               # 3x1

rpy_next = world_pose[3:6] + euler_dot_star * dt



x_world_ref_next = ca.vertcat(p_world_next, rpy_next)
q_ref_next       = q + nu_star[6:6+n_joints] * dt

# Pack state = [world_pose; q]
state0 = ca.vertcat(world_pose, q)  # (6+n) x 1

# One IK step as a CasADi function (state -> next_state)
state1 = ca.vertcat(x_world_ref_next, q_ref_next)  # (6+n) x 1

e_p_task_star = Jp_ee_W @ nu_star - v_des
e_axis_task_star = Jw_ee_W @ nu_star - w_des_axis

ik_step = ca.Function(
    'ik_step',
    [state0, kp, p_des, w_rp, w_reg, k_rp, a_des_x, a_des_z, k_axis, w_axis, w_align, k_align, dt, base_pose, tip_offset_pose, q_lower, q_upper, dq_max],
    [state1, e_p_task_star, e_axis_task_star]
)

In [ ]:
# Keep the accumulated solver as an MX call graph (avoid expanding 5000 QPs).
for name in ('q', 'world_pose', 'kp', 'p_des', 'w_rp', 'w_reg', 'k_rp',
             'a_des_x', 'a_des_z', 'k_axis', 'w_axis', 'w_align', 'k_align',
             'dt', 'base_pose', 'tip_offset_pose', 'q_lower', 'q_upper', 'dq_max'):
    old = globals()[name]
    globals()[name] = ca.MX.sym(name, old.size1(), old.size2())
state0 = ca.vertcat(world_pose, q)
# Mapaccum over N iterations
N = 5000
ik_step_acc = ik_step.mapaccum('ik_step_acc', N)

# Accumulated trajectories
state_traj, e_p_traj, e_axis_traj = ik_step_acc(
    state0, kp, p_des, w_rp, w_reg, k_rp, a_des_x, a_des_z, k_axis, w_axis, w_align, k_align, dt, base_pose, tip_offset_pose, q_lower, q_upper, dq_max
)

In [ ]:
# Take the final values (last column)
stateN = state_traj[:, -1]
x_world_next = stateN[0:6]
q_next       = stateN[6:6+n_joints]

e_p_task_star_new    = e_p_traj[:, -1]
e_axis_task_star_new = e_axis_traj[:, -1]

ik_func = ca.Function(
    'bounded_task_based_ik',
    [q, world_pose, kp, p_des, w_rp, w_reg, k_rp,  a_des_x, a_des_z, k_axis, w_axis, w_align, k_align, dt, base_pose, tip_offset_pose, q_lower, q_upper, dq_max],
    [x_world_next, q_next, e_p_task_star_new, e_axis_task_star_new]
)

In [ ]:
q_val=[1.0002, 1.9335, 0.3438, 0.0125]
world_pose_val=[ 4.1448e-01 ,-1.5551e-01, -1.3213e+00,  3.7661e-05,  3.1087e-04 , 9.7403e-01]
p_des_val=[ 5.5038e-01 ,-1.2290e-17 ,-1.5958e+00]
a_des_z_val = ca.DM([0, 0, -1])   # tool z axis points downward
a_des_x_val = ca.DM([1, 0, 0])    # tool x axis aligned with world x


kp_val = np.array([1.0, 1.0, 1.0], dtype=float)
w_rp_val = 1.0
w_reg_val = 0.02
k_rp_val = 0.2
k_axis_val = 1.0
w_axis_val = 1.5
dt_val = 1.0 / 500.0
w_align_val = 1.0
k_align_val = 1.0

base_pose_val = np.asarray([0.190, 0.000, -0.120, 3.141592653589793, 0.000, 0.000], dtype=float)
tip_offset_pose_val = np.asarray([0.00, 0.00, 0.04, 0.00, 0.00, 0.00], dtype=float)

ik_func(q_val, world_pose_val, kp_val, p_des_val, w_rp_val, w_reg_val, k_rp_val,
         a_des_x_val, a_des_z_val,
         k_axis_val, w_axis_val, w_align_val, k_align_val,
           dt_val, base_pose_val, tip_offset_pose_val,
           [0.01]*4, [6.27, 3.4, 3.4, 5.7], [0.3]*4)

### Bounded differential IK comparison
This differential solver is retained for notebook comparison only; it is not exported. It is not used by the current simulator. Runtime must supply joint bounds from the deployed robot description. Endpoint FK tolerances must be checked separately; velocity residuals are not pose errors. This local solver does not guarantee global convergence or collision-free motion, and its orientation error has a stationary point at 180 degrees.

The geometric Jacobian now maps physical world-frame base twist plus joint rates. The comparison integrates Euler coordinates with the matching URDF RPY rate map; that integration remains singular at pitch ±90 degrees. The active quaternion endpoint solver does not use this comparison.


## Endpoint IK formulation and export
Set `fk_path` to your exported arm FK model and choose a new `export_dir`. Exports are staged locally; installation into a simulator workspace is a separate step. Only IPOPT is exported; SQP remains an analysis option.


In [ ]:
from pathlib import Path
import casadi as ca

def build_endpoint_ik(fk_path, plugin='ipopt'):
    fk = ca.Function.load(str(fk_path))
    z = ca.MX.sym('configuration', 11)  # world xyz, unit quaternion wxyz, joints
    p = ca.MX.sym('task', 31)  # measured(11), target(7), base(6), tip(6), alignment weight
    link_poses = fk(z[7:11], p[18:24], ca.DM.zeros(6), p[24:30])
    local = link_poses[-1]
    # Original preference: first arm-link +X points along vehicle -X.
    # Evaluate in the vehicle frame; common world rotation cancels. A directed
    # dot-product cost avoids the cross-product's false zero at opposite alignment.
    # alpha_axis_e is the first revolute joint, with identity origin rotation
    # and local Z axis (alpha_5_robot.urdf). Use its rotation directly: the
    # legacy FK matrix-to-quaternion chart has singular derivatives here.
    roll, pitch, yaw = p[21], p[22], p[23]
    cx = ca.cos(yaw)*ca.cos(pitch)
    cy = ca.cos(yaw)*ca.sin(pitch)*ca.sin(roll)-ca.sin(yaw)*ca.cos(roll)
    alignment_cost = 1 + cx*ca.cos(z[7]) + cy*ca.sin(z[7])
    qb = z[3:7]
    w, v = qb[0], qb[1:]
    # Quaternion rotation, constrained to SO(3) by unit norm below.
    rotation = (w*w-ca.dot(v,v))*ca.MX.eye(3) + 2*ca.mtimes(v,v.T) + 2*w*ca.skew(v)
    position = z[:3] + rotation @ local[:3]
    ql = local[3:7] / ca.norm_2(local[3:7])
    actual_q = ca.vertcat(w*ql[0]-ca.dot(v,ql[1:]),
                         w*ql[1:]+ql[0]*v+ca.cross(v,ql[1:]))
    target_q = p[14:18] / ca.norm_2(p[14:18])
    rotation_error = (target_q[0]*actual_q[1:] - actual_q[0]*target_q[1:]
                      - ca.cross(target_q[1:], actual_q[1:]))
    constraints = ca.vertcat(position-p[11:14], rotation_error, ca.dot(qb,qb)-1)
    # Both quaternion signs describe the same rotation: objective is sign invariant.
    objective = (ca.sumsqr(z[:3]-p[:3]) + .02*ca.sumsqr(z[7:]-p[7:11])
                 + .08*(1-ca.dot(qb,p[3:7])**2) + (1-rotation[2,2])
                 + p[30]*alignment_cost)
    # Parameter sensitivities are not used by IK; avoid differentiating the
    # legacy FK quaternion chart with respect to fixed mounting parameters.
    common = {'print_time': False, 'error_on_fail': False, 'calc_lam_p': False}
    if plugin == 'ipopt':
        common.update({'ipopt.print_level':0, 'ipopt.sb':'yes',
                       'ipopt.max_iter':200, 'ipopt.max_cpu_time':.2,
                       'ipopt.tol':1e-7, 'ipopt.constr_viol_tol':1e-7,
                       'ipopt.hessian_approximation':'limited-memory', 'ipopt.bound_relax_factor':0.})
    else:
        common.update({'max_iter':100, 'print_header':False, 'print_iteration':False,
                       'print_status':False, 'qpsol':'qrqp',
                       'qpsol_options':{'print_header':False,'print_iter':False,'error_on_fail':False},
                       'hessian_approximation':'limited-memory',
                       'tol_pr':1e-7,'tol_du':1e-7})
    return ca.nlpsol('endpoint_ik_'+plugin,plugin,{'x':z,'p':p,'f':objective,'g':constraints},common)


In [ ]:
# User-configurable paths; no implicit writes into a ROS workspace.
fk_path = Path('fk_eval.casadi').resolve()
export_dir = Path('generated_kinematics').resolve()
solver = build_endpoint_ik(fk_path, 'ipopt')
export_dir.mkdir(parents=True, exist_ok=False)
output = export_dir / 'endpoint_ik_ipopt.casadi'
solver.save(str(output))
print(output)


## Tool-wrench geometric Jacobian export
Exports the corrected `geo_J_full` at the IK tool origin. Rows are world-frame linear/angular velocity; columns are world-frame base linear/angular velocity followed by joint rates. SimLab rotates base input columns to its FRD body convention before applying the transpose. The transform output supplies the exact tool point/orientation, including mounting and tip offsets.


In [ ]:
# Export the corrected geometric Jacobian; do not differentiate Euler angles.
from pathlib import Path
import sys
import casadi as ca
source_root = Path.cwd().parent if Path.cwd().name == 'usage' else Path.cwd()
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
from system.robot import RobotDynamics
model = RobotDynamics(use_jit=False)
model.from_file(str(source_root/'usage/urdf/reach_alpha_5/alpha_5_robot.urdf'))
kinematics = model._kinematics('base_link', 'alpha_standard_jaws_base_link', floating_base=True)
parameters = kinematics['parameters']
joints, mounting, world, tip = [parameters[i] for i in (10, 14, 15, 16)]
tool_geometric = ca.Function('tool_geometric_jacobian', [joints, mounting, world, tip],
    [kinematics['geo_J_full'][-1], kinematics['i_X_0s'][-1]],
    ['joints', 'mounting', 'world', 'tip'], ['jacobian', 'transform'])
output = export_dir / 'tool_geometric_jacobian.casadi'
if output.exists():
    raise FileExistsError(output)
tool_geometric.save(str(output))
print(output)
